# Experiment 4 — Tactical Deconfliction with a Dynamic Geofence

A keep-out geofence is activated while the UAV is already inside it. The tactical pipeline must detect the new airspace conflict and issue a lateral escape route.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from musas.airspace import Airspace, Cylinder3D, Geofence
from musas.algorithms import (
    FirstOrderDynamics,
    FlightPlanAwareTracking,
    HybridGeometricManeuver,
    MinimumSeparationTacticalConflictPrediction,
    WaypointGuidance,
)
from musas.domain import FlightPlan, Waypoint
from musas.simulation import (
    Simulation, SimulationDataset, VisualizationConfig,
    animate_dataset, calculate_metrics, plot_dataset,
)

OUTPUT_DIRECTORY = Path("outputs") / "output_exp4"
DATASET_DIRECTORY = OUTPUT_DIRECTORY / "datasets"
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
DATASET_DIRECTORY.mkdir(exist_ok=True)

In [ ]:
SIMULATION_END = 45.0
GEOFENCE_ACTIVATION = 14.0
MINIMUM_SEPARATION_DISTANCE = 30.0
COLLISION_DISTANCE = 5.0

flight_plan = FlightPlan([
    Waypoint(-100.0, 0.0, 100.0, 2.0),
    Waypoint(100.0, 0.0, 100.0, 32.0),
])
dynamic_geofence = Geofence(
    geofence_id="dynamic-zone",
    name="Dynamic keep-out zone",
    geometry=Cylinder3D(-20.0, 0.0, 25.0, 80.0, 120.0),
)

In [ ]:
dataset_path = DATASET_DIRECTORY / "dynamic_geofence.sqlite"
tactical_predictor = MinimumSeparationTacticalConflictPrediction(
    minimum_separation_distance=MINIMUM_SEPARATION_DISTANCE,
    maximum_time_difference=0.25,
)
simulation = Simulation(
    airspace=Airspace(),
    tracking_algorithm=FlightPlanAwareTracking(),
    tactical_conflict_prediction_algorithm=tactical_predictor,
    tactical_deconfliction_algorithm=HybridGeometricManeuver(
        conflict_prediction_algorithm=tactical_predictor,
        minimum_separation_distance=MINIMUM_SEPARATION_DISTANCE,
        airspace_clearance=10.0,
        maximum_speed=15.0,
        cooldown=8.0,
    ),
    report_interval=0.5,
    history_window=6,
    prediction_horizon=14,
    prediction_step=0.5,
    tactical_assessment_interval=0.25,
    flight_update_interval=0.1,
    trajectory_record_interval=0.2,
    dataset_path=dataset_path,
    overwrite_dataset=True,
    dataset_metadata={"experiment": "dynamic_geofence"},
)
simulation.schedule_geofence(dynamic_geofence, GEOFENCE_ACTIVATION)
simulation.add_uav(
    name="UAV 1",
    flight_plan=flight_plan,
    dynamics_algorithm=FirstOrderDynamics(100.0, 0.0),
    guidance_algorithm=WaypointGuidance(min_speed=0.0, max_speed=15.0),
    waypoint_acceptance_radius=1.0,
)
simulation.run(until=SIMULATION_END)
simulation.close()
dataset_path

In [ ]:
with SimulationDataset(dataset_path) as dataset:
    metrics = calculate_metrics(
        dataset, MINIMUM_SEPARATION_DISTANCE, COLLISION_DISTANCE,
    )
{key: metrics[key] for key in (
    "detected_tactical_conflicts",
    "tactical_interventions",
    "geofence_violations",
    "collision_events",
)}

In [ ]:
config = VisualizationConfig(
    uav_names=("UAV 1",), view="xy",
    show_real_trajectories=True, show_current_positions=True,
    show_flight_plans=True, show_tracks=True, show_geofences=True,
    show_wind_zones=False, show_obstacles=False,
    figure_size=(8.0, 5.5), frame_step=0.5,
    title="TD with a dynamic geofence",
)
for timestamp in (12.0, 14.0, 18.0, 26.0):
    image_path = OUTPUT_DIRECTORY / f"t_{timestamp:04.1f}.png"
    figure = plot_dataset(
        dataset_path, config, timestamp, image_path, dpi=180,
    )
    display(figure)

animation_path = OUTPUT_DIRECTORY / "dynamic_geofence.gif"
animate_dataset(
    dataset_path, config, animation_path, fps=6.0, dpi=100,
)
display(Image(filename=animation_path))